In [ ]:
import os
import cv2
import shutil
import random
import numpy as np 
import pandas as pd

import torch
import torch.nn as nn
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import albumentations as A

import segmentation_models_pytorch as smp

from sklearn.mixture import GaussianMixture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
set_seed(0)

In [ ]:
def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            folder_name = os.path.basename(dirname)
            full_name = f"{filename.split('.')[0]}"
            name.append(full_name)
    return pd.DataFrame({'id': name}, index=np.arange(0, len(name)))

In [ ]:
class CloudDataset(Dataset):
    def __init__(self, img_path, X, Y=None, mean=None, std=None, transform=None):
        self.img_path = img_path
        self.X = X
        self.Y = Y
        self.transform = transform
        self.mean = mean
        self.std = std
        self.has_label = Y is not None

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = Image.fromarray(self.transform(image=img)['image'])
        else:
            img = Image.fromarray(img)
        
        t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        img = t(img)
        
        if self.has_label:
            label = self.Y[idx]
            return img, label
        else:
            return img

In [ ]:
batch_size = 1
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [ ]:
t_train = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA)])
t_valid = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA)])

In [ ]:
IMAGE_PATH_TRAIN = '../target_train/'
df_train = create_df(IMAGE_PATH_TRAIN)
df_train_radiation = pd.read_csv('./radiation_train.csv')
X_train = df_train['id'].values
Y_train = df_train_radiation['Radiation(Ldown-Lup)(Wm^2)'].values
train_set = CloudDataset(IMAGE_PATH_TRAIN, X_train, Y_train, mean, std, t_train)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

In [ ]:
base_model = smp.Unet('timm-mobilenetv3_large_100', encoder_weights='imagenet', classes=9,
                 activation=None, encoder_depth=5, decoder_channels=[256, 128, 64, 32, 16])

In [ ]:
class ModelWithIntermediate(nn.Module):
    def __init__(self, encoder, classifier):
        super(ModelWithIntermediate, self).__init__()
        self.encoder = encoder
        self.pool = classifier[0]
        self.flatten = classifier[1]
        self.fc1 = classifier[2]  # nn.Linear(960, reduced_dimension)
        # self.fc2 = classifier[3]  # removed this layer

    def forward(self, x):
        features = self.encoder(x)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        x = self.pool(features)
        x = self.flatten(x)
        reduced_features = self.fc1(x)  # run to here
        return reduced_features

In [ ]:
encoder = base_model.encoder
encoder_out_channels = 960
reduced_dimension = 6

classifier = nn.Sequential(
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(encoder_out_channels, reduced_dimension),
    nn.Linear(reduced_dimension, 1)
)

model = ModelWithIntermediate(encoder, classifier)

In [ ]:
checkpoint = torch.load("6_best.pth", map_location="cpu")
encoder_state_dict = {k.replace("encoder.model.", ""): v for k, v in checkpoint.items() if k.startswith("encoder.model.")}
classifier_state_dict = {k: v for k, v in checkpoint.items() if k.startswith("classifier.2")}
model.encoder.load_state_dict(encoder_state_dict, strict=True)
model.fc1.load_state_dict({
    "weight": classifier_state_dict["classifier.2.weight"],
    "bias": classifier_state_dict["classifier.2.bias"]
})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.eval()

# --------------- train_loader ---------------
train_all_features = []
train_all_labels = []

with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        feats = model(imgs)
        train_all_labels.append(labels.cpu())
        train_all_features.append(feats.cpu())
        
train_all_features = torch.cat(train_all_features, dim=0).numpy()
train_all_labels = torch.cat(train_all_labels, dim=0).numpy()  # (N_train, feature_dim)

In [ ]:
n_components = 3
gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
cluster_labels = gmm.fit_predict(train_all_features)

In [ ]:
IMAGE_PATH_ALL = '../target_valid/'
df_all = create_df(IMAGE_PATH_ALL)
X_all = df_all['id'].values

all_set = CloudDataset(img_path=IMAGE_PATH_ALL, X=X_all, mean=mean, std=std, transform=t_valid)
all_loader = DataLoader(all_set, batch_size=batch_size, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.eval()
all_features = []
with torch.no_grad():
    for imgs in all_loader:
        imgs = imgs.to(device)
        feats = model(imgs)
        all_features.append(feats.cpu())
all_features = torch.cat(all_features, dim=0).numpy()

In [ ]:
all_cluster_labels = gmm.predict(all_features)

OUTPUT_DIR = '../clustered_images_valid/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for img_name, cluster_label in zip(X_all, all_cluster_labels):
    src_path = os.path.join(IMAGE_PATH_ALL, img_name + '.jpg')
    cluster_folder = os.path.join(OUTPUT_DIR, f'cluster_{cluster_label+1}')
    os.makedirs(cluster_folder, exist_ok=True)

    dst_path = os.path.join(cluster_folder, img_name + '.jpg')
    shutil.copyfile(src_path, dst_path)